<a href="https://colab.research.google.com/github/samridhi-gupta08/Project-Minerva/blob/main/Samridhi_250968_BCS_Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

url = "https://www.unwomen.org/en/articles/faqs/faqs-afghanistan"

r = requests.get(url)

soup = BeautifulSoup(r.text, "html.parser")

title = soup.find("h1").text.strip()

paras = soup.find_all("p")

text = ""

for p in paras:
    t = p.get_text().strip()

    if t != "":
        text += t + "\n"

print(title)
print(text[:1000])

with open("article.txt", "w", encoding="utf-8") as f:
    f.write(text)

data = {
    "title": title,
    "content": text
}

with open("article.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

In [ ]:
import nltk
import spacy
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

with open("article.txt", "r", encoding="utf-8") as f:
    text = f.read()

sentences = sent_tokenize(text)

words = word_tokenize(text)

print("sentences:", len(sentences))
print("words:", len(words))

print(sentences[:3])
print(words[:20])

nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

filtered = []

for token in doc:
    if not token.is_stop and not token.is_punct:
        filtered.append(token.text)

print(filtered[:40])

lemmas = []

for token in doc:
    lemmas.append(token.lemma_)

print(lemmas[:40])

In [ ]:
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

with open("article.txt", "r", encoding="utf-8") as f:
    text = f.read()

sents = sent_tokenize(text)

chunks = []

size = 5

for i in range(0, len(sents), size):
    chunk = " ".join(sents[i:i+size])
    chunks.append(chunk)

print("chunks:", len(chunks))

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print(embeddings.shape)

In [ ]:
!pip install chromadb
!pip install rfc3987
import chromadb
import pickle

from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt")

with open("article.txt", "r", encoding="utf-8") as f:
    text = f.read()

sents = sent_tokenize(text)

parts = []

for i in range(0, len(sents), 5):
    parts.append(" ".join(sents[i:i+5]))

model = SentenceTransformer("all-MiniLM-L6-v2")

vecs = model.encode(parts)

client = chromadb.Client()

collection = client.create_collection("afghan_data")

collection.add(
    documents=parts,
    embeddings=vecs.tolist(),
    ids=[str(i) for i in range(len(parts))]
)

print("done")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()

collection = client.get_collection("afghan_data")

q = "women not allowed to study"

q_embed = model.encode([q]).tolist()

result = collection.query(
    query_embeddings=q_embed,
    n_results=2
)

print(result["documents"])

In [ ]:
#requirements.txt
requests
beautifulsoup4
nltk
spacy
sentence-transformers
chromadb